# Práctica 3. Black-Litterman y paridad de riesgo jerárquica

Markowitz produce carteras inestables y concentradas, por eso el objetivo de esta práctica es explorar las alternativas propuestas en los modelos de **Black-Litterman** y **HRP**. A lo largo de este notebook vamos a recorrer los diferentes pasos que hemos realizado para implementar estas técnicas incluyendo las diferentes decisiones que hemos tomado en referencia al universo de activos, mandato de inversión y cartera de mercado.


In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 0. Mandato

El primer paso de la práctica era decidir un mandato y un universo de activos. Para hacerlo de la forma más profesional y realista posible, hemos investigado por nuestra cuenta los diferentes formatos y hemos decidido utilizar el IPS (Investment Policy Statement) tal y como lo estructura el CFA Institute.

| | Elemento | Mandato |
|---|---|---|
| **Objetivos** | Rentabilidad | Crecimiento del capital a largo plazo; batir al benchmark en ~2-3 % anual |
| | Riesgo | Volatilidad objetivo de ~12-15 % anual; caída máxima tolerable de ~30 % |
| **Restricciones** | Horizonte | 10 años |
| | Impuestos | No se consideran (simplificación académica) |
| | Liquidez | Sin necesidades de liquidez a corto plazo |
| | Legal | Solo ETF cotizados y líquidos; solo largos; sin derivados ni apalancamiento |
| | Circunstancias particulares | Ninguna |
| **Política** | Moneda base | USD |
| | Clases de activo | Renta variable y renta fija |
| | Benchmark | Cartera de mercado de nuestro universo: los 11 ETF con los pesos w_mkt fijos (Doeswijk, cierre de 2012; apartado 2), rebalanceada trimestralmente |
| | Rebalanceo | Trimestral |

Para el universo de activos hemos elegido dos clases, renta variable y renta fija. Dentro de cada una buscábamos un conjunto de ETF que representara más o menos el mercado mundial. El problema es que Black-Litterman necesita además los pesos de ese mercado, y conseguir unos pesos realistas ha sido bastante más difícil de lo que parecía. Al final la elección del universo y la de los pesos han ido de la mano.

Lo primero que probamos fue usar los activos bajo gestión de los ETF. Es el dato más fácil de conseguir, pero mide cuánto dinero ha entrado en cada fondo además de cuánto se han revalorizado sus activos y no cuánto vale cada mercado. Los ETF son un vehículo muy habitual para invertir en acciones y mucho menos para invertir en bonos, así que la renta fija salía infraponderada. El sesgo también aparecía dentro de cada clase. Con los activos de hoy, SPY e IWM suman el 89 % de la renta variable del universo, cuando EE. UU. pesa en torno al 64 % del MSCI ACWI. BWX no llega al 2 % de la deuda pública del universo, y la deuda pública de fuera de EE. UU. es más de la mitad del total mundial. Los ETF que cotizan en EE. UU. los compran sobre todo inversores estadounidenses, y algunos, como SPY, TLT o HYG, se usan mucho para operar a corto plazo.

La alternativa era la capitalización de mercado de los índices replicados en el caso de la renta variable o el valor de mercado en el caso de los instrumentos de renta fija. La mayoría de estos datos son de difícil acceso o de pago, por lo que no eran una opción a la hora de sacar los pesos de la cartera de mercado. Finalmente hemos decidido usar como referencia la cartera de mercado global multiactivo de Doeswijk, Lam y Swinkels, publicada en el Financial Analysts Journal (2014) con el título «The Global Multi-Asset Market Portfolio, 1959–2012». Los autores estiman el valor de mercado de cada clase de activo a escala mundial a partir de los principales índices de referencia, y mantienen una actualización anual de los datos hasta 2023 (Doeswijk, Lam y Swinkels, 2024). 


Quedaba decidir si los pesos debían cambiar con el tiempo. Probamos a actualizarlos con los precios a partir de 2012, y en 2023 la renta variable salía con un 64 % cuando en el paper es un 49 %. Los precios no recogen la deuda nueva que emiten los gobiernos cada año, y el error crece unos 1,3 puntos por año. Actualizar con cada foto anual del paper sí funcionaba, pero a partir de 2023 no hay más datos. Al mirar la serie vimos además que el reparto entre clases cambia poco. Desde 2013 la renta variable se mueve entre el 43 % y el 49 % del total.

Por eso usamos una foto fija, la de cierre de 2012. Es la última anterior al inicio del backtest fuera de muestra, a mediados de 2013, así que no usa información posterior. El paper se publicó en 2014, pero los datos son los de cierre de 2012. Los pesos no cambian, pero las rentabilidades de equilibrio sí, porque la matriz de covarianzas se vuelve a estimar en cada rebalanceo. Para que los pesos del paper encajen con nuestros activos sin inventar repartos, elegimos los ETF de forma que cada uno corresponda a una fila de la cartera de Doeswijk. Esto tiene un coste. El universo es menos detallado de lo que nos habría gustado, y algunos ETF solo cubren la parte estadounidense de su fila. A cambio, la cartera de mercado es coherente, se puede comprobar con los datos del paper y depende de muy pocos supuestos externos.

La moneda base es el dólar. Todos los ETF del universo cotizan en dólares y los datos de la cartera de mercado de Doeswijk también están en dólares, así que no hace falta convertir nada. Con el euro como moneda base, la rentabilidad de toda la cartera dependería además del tipo de cambio, y ese es un riesgo que el mandato no busca.

Pedimos batir al benchmark en un 2-3 % anual. Para una gestión activa es un objetivo moderado. La cartera es solo larga y sin apalancamiento por simplificar la práctica.

## 1. Universo

Once ETF en dos clases de activo, con historia común desde julio de 2010. La fecha la marca EMLC, que es el que empieza a cotizar más tarde.

| Clase | Fila de Doeswijk | Ticker | Qué es | Desde |
|---|---|---|---|---|
| Renta variable | MSCI ACWI: EE. UU. | SPY | S&P 500 | 1993 |
| | MSCI ACWI: desarrollados ex-EE. UU. | EFA | MSCI EAFE (Europa, Australasia y Lejano Oriente) | 2001 |
| | MSCI ACWI: emergentes | EEM | MSCI Emerging Markets | 2003 |
| Renta fija | Deuda pública: EE. UU. | IEF | Tesoro 7-10 años | 2002 |
| | Deuda pública: EE. UU. | TLT | Tesoro 20+ años | 2002 |
| | Deuda pública: ex-EE. UU. | BWX | Soberanos ex-EE. UU. en moneda local, sin cubrir | 2007 |
| | Crédito investment grade | LQD | Corporativa IG de EE. UU. | 2002 |
| | High yield | HYG | High yield de EE. UU. | 2007 |
| | Ligados a inflación | TIP | TIPS de EE. UU. | 2003 |
| | Deuda emergente: soberana en USD | EMB | J.P. Morgan EMBI Global Core | 2007 |
| | Deuda emergente: soberana en moneda local | EMLC | J.P. Morgan GBI-EM Global Core | 2010 |

El universo sale de la cartera de mercado global de Doeswijk, Lam y Swinkels (2014). Cada ETF representa una fila de esa cartera, de modo que su peso de mercado se obtiene casi directamente de sus datos. Solo hacen falta tres números de fuera para repartir las filas que tienen más de un ETF.

Antes de tomar esta decisión probamos un universo de 16 ETF, con las regiones de renta variable por separado (VGK, EWJ, EWC y EPP), las small caps de EE. UU. y del resto del mundo (IWM y SCZ) y la deuda hipotecaria (MBB). Era más rico y detallado, pero para estimar los pesos de mercado hacían falta unos varios supuestos que nos parecían imprecisos. Nuestro criterio ha sido escoger un universo algo más grueso a cambio de que la cartera de mercado sea coherente y fácil de estimar. De este modo, nos quedamos solo con renta variable y renta fija, que en 2012 suman el 91 % de la cartera de Doeswijk. 

La renta variable va en tres bloques, EE. UU. (SPY), desarrollados fuera de EE. UU. (EFA) y emergentes (EEM). Es el reparto natural del MSCI ACWI, y para separarlo basta con dos números, el peso de EE. UU. y el de los emergentes en el índice. Con las regiones por separado haría falta el peso de cada una. SPY replica el S&P 500 y deja fuera las empresas medianas de EE. UU. que sí están en el ACWI. Lo aceptamos porque es el ETF más líquido del mundo y el S&P 500 se mueve casi igual que el mercado estadounidense completo. El índice de EFA no incluye Canadá. Su peso en el ACWI se suma al de EFA, que es el bloque más parecido.

En renta fija el universo sigue las filas de Doeswijk. La deuda pública se reparte entre el Tesoro de EE. UU. (IEF y TLT) y la deuda pública del resto del mundo (BWX). IEF y TLT son dos tramos del mismo emisor y se mueven de forma muy parecida. Los dejamos a propósito, porque dos activos casi redundantes son justo el caso en el que Markowitz se vuelve inestable y en el que HRP debería agruparlos. BWX no cubre la divisa. La moneda forma parte del riesgo de invertir en bonos de otros países, y la cartera de mercado medida en dólares también la lleva sin cubrir.

LQD, HYG y TIP representan filas que en Doeswijk son mundiales, el crédito investment grade, el high yield y los bonos ligados a la inflación. Los tres contienen solo bonos de EE. UU. No hemos encontrado ETF que cubran estas clases en todo el mundo con historia desde 2012, y EE. UU. es la parte más grande de cada una. La fila de crédito investment grade incluye además la deuda hipotecaria. No la separamos con MBB para no añadir otro supuesto de reparto, así que LQD representa la fila entera.

La deuda emergente entra con dos ETF, EMB para los bonos soberanos en dólares y EMLC para los soberanos en moneda local. Las dos filas existen por separado en Doeswijk y se comportan de forma distinta, porque EMLC lleva además el riesgo de las divisas emergentes. EMLC empieza a cotizar en julio de 2010 y acorta la historia común en dos años y medio. Aun así quedan más de dieciséis años de datos, suficientes para la ventana de tres años del backtest.

No incluimos ETF monetarios. El mandato no tiene necesidades de liquidez y la liquidez no forma parte de la cartera de Doeswijk. Tampoco hay ETF de factores ni de sectores. Son formas distintas de repartir las mismas acciones que ya están en SPY, EFA y EEM, y en la cartera de mercado no tienen un peso propio.

Descarga de precios y comprobación de la historia de cada activo.

In [2]:
from quantmgmt.data.prices import download_prices
from quantmgmt.risk.returns import to_returns

TICKERS = ["SPY", "EFA", "EEM", "IEF", "TLT", "BWX", "LQD", "HYG", "TIP", "EMB", "EMLC"]
INICIO, FIN = "2010-07-01", "2026-10-01"   # FIN excluido: último día = 30-sep-2026

precios = download_prices(TICKERS, start=INICIO, end=FIN)

# Primera fecha con precio de cada ETF
inicios = precios.apply(lambda s: s.first_valid_index())
print(inicios)

# Historia común: desde que cotizan todos
precios = precios.loc[inicios.max():]
print("Huecos:", precios.isna().sum().sum())

rent = to_returns(precios)
print(rent.index[0].date(), rent.index[-1].date(), rent.shape)


SPY    2010-07-01
EFA    2010-07-01
EEM    2010-07-01
IEF    2010-07-01
TLT    2010-07-01
BWX    2010-07-01
LQD    2010-07-01
HYG    2010-07-01
TIP    2010-07-01
EMB    2010-07-01
EMLC   2010-07-23
dtype: datetime64[us]
Huecos: 0
2010-07-26 2026-09-30 (4071, 11)


In [3]:
corr = rent.corr()
corr.round(2).style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1).format("{:.2f}")


,SPY,EFA,EEM,IEF,TLT,BWX,LQD,HYG,TIP,EMB,EMLC
SPY,1.00,0.85,0.77,-0.23,-0.27,0.16,0.21,0.76,-0.04,0.53,0.50
EFA,0.85,1.00,0.84,-0.20,-0.25,0.35,0.21,0.73,-0.00,0.55,0.65
EEM,0.77,0.84,1.00,-0.16,-0.21,0.31,0.19,0.66,0.01,0.54,0.67
IEF,-0.23,-0.20,-0.16,1.00,0.91,0.50,0.72,-0.00,0.79,0.33,0.12
TLT,-0.27,-0.25,-0.21,0.91,1.00,0.40,0.69,-0.08,0.73,0.28,0.04
BWX,0.16,0.35,0.31,0.50,0.40,1.00,0.49,0.32,0.46,0.47,0.57
LQD,0.21,0.21,0.19,0.72,0.69,0.49,1.00,0.43,0.67,0.62,0.39
HYG,0.76,0.73,0.66,-0.00,-0.08,0.32,0.43,1.00,0.14,0.67,0.59
TIP,-0.04,-0.00,0.01,0.79,0.73,0.46,0.67,0.14,1.00,0.45,0.23
EMB,0.53,0.55,0.54,0.33,0.28,0.47,0.62,0.67,0.45,1.00,0.67


# Parte A. Black-Litterman

Implementación en `quantmgmt/optimizers/black_litterman.py`.

## 2. Pesos de equilibrio

Definimos el mercado de nuestro universo con la foto de cierre de 2012 de la cartera de Doeswijk, Lam y Swinkels. En el apartado 0 explicamos por qué usamos el valor de mercado y no los activos bajo gestión, y por qué una foto fija en lugar de pesos que cambien con el tiempo. Ahora nos queda explicar cómo pasamos de las filas del paper a los once ETF.

Lo hacemos en dos niveles. El primero da el peso de cada clase de activo y sale entero del paper. El segundo reparte cada clase entre sus ETF cuando hay más de uno, y para eso usamos dos fuentes más, el Tesoro de EE. UU. para la deuda pública y el ETF del MSCI ACWI para la renta variable.

Doeswijk mide cada clase con el valor de mercado de un índice de referencia. Quitamos el private equity y el inmobiliario, que no están en el universo y suman un 8,7 % del total, y reescalamos el resto para que sume uno.

| Clase | Índice que usa Doeswijk | Valor en 2012 (billones de USD) | Peso reescalado | ETF |
|---|---|---|---|---|
| Renta variable | MSCI ACWI más MSCI ACWI Small Cap, sin el inmobiliario cotizado | 32,9 | 39,8 % | SPY, EFA y EEM |
| Deuda pública | Bloomberg Multiverse Government | 26,7 | 32,3 % | IEF, TLT y BWX |
| Crédito investment grade | Bloomberg Multiverse sin deuda pública ni high yield | 16,8 | 20,3 % | LQD |
| Deuda emergente | J.P. Morgan GBI-EM, EMBI Global y CEMBI Broad, y Bloomberg EM Inflation-Linked | 2,7 | 3,2 % | EMB y EMLC |
| Ligados a la inflación | Bloomberg Global Aggregate Inflation-Linked | 2,1 | 2,5 % | TIP |
| High yield | Bloomberg Global Corporate High Yield | 1,5 | 1,8 % | HYG |

Las tres últimas filas tienen un solo ETF en nuestro universo y pasan directas.

El segundo nivel, consiste en las distribuciones de pesos de la categoráis generales entre cada ETF de nuestro unvierso.

| Clase | Reparto | Dato | Fuente |
|---|---|---|---|
| Deuda pública | EE. UU. (IEF y TLT) frente al resto (BWX) | EE. UU. 32 %, resto 68 % | Notes y bonds del Tesoro a diciembre de 2012 (8,6 billones), Monthly Statement of the Public Debt, frente a los 26,7 billones de Doeswijk |
| Deuda pública de EE. UU. | IEF frente a TLT | Notes 85,5 %, bonds 14,5 % | Mismo informe del Tesoro (7,3 y 1,2 billones) |
| Renta variable | SPY, EFA y EEM | EE. UU. 45,9 %, desarrollados fuera de EE. UU. con Canadá 41,3 %, emergentes 12,8 % | Cartera por países del ETF iShares MSCI ACWI a 31 de enero de 2013, informe N-CSRS presentado a la SEC |
| Deuda emergente | EMLC frente a EMB | Moneda local 55 %, dólares 45 % | Subfilas de Doeswijk. Moneda local e inflación (0,95 y 0,53 billones) a EMLC, dólares y corporativa (0,58 y 0,62) a EMB |

En la deuda pública contamos solo las notes y los bonds, porque los índices de Doeswijk solo incluyen bonos a más de un año. El dato es aproximado, porque compara el nominal del Tesoro con el valor de mercado del índice, y en 2012 los bonos cotizaban algo por encima del nominal. Las notes van de dos a diez años y no tenemos un ETF del tramo corto, así que IEF recoge todo ese tramo.

En la renta variable, el MSCI ACWI no publica gratis sus pesos históricos por país, así que usamos la cartera del ETF que lo replica. Su informe más cercano al cierre de 2012 es el de enero de 2013. En el anterior, de octubre de 2012, EE. UU. pesaba un 46,7 %, así que el dato de diciembre queda entre los dos y el error es de menos de medio punto de cartera. Separamos emergentes y desarrollados con la clasificación de MSCI de esa fecha. Canadá no está en EFA y su peso se suma a ese bloque, que es el más parecido. Las small caps se reparten en las mismas proporciones.

En la deuda emergente, la corporativa está casi toda en dólares y por eso va con EMB.

### Resultado

| Activo | Peso de mercado | Cálculo |
|---|---|---|
| SPY | 18,3 % | Renta variable × EE. UU. |
| EFA | 16,4 % | Renta variable × desarrollados fuera de EE. UU. |
| EEM | 5,1 % | Renta variable × emergentes |
| IEF | 8,9 % | Deuda pública × EE. UU. × notes |
| TLT | 1,5 % | Deuda pública × EE. UU. × bonds |
| BWX | 22,0 % | Deuda pública × resto del mundo |
| LQD | 20,3 % | Crédito investment grade |
| HYG | 1,8 % | High yield |
| TIP | 2,5 % | Ligados a la inflación |
| EMB | 1,5 % | Deuda emergente × dólares |
| EMLC | 1,8 % | Deuda emergente × moneda local |

Los pesos están redondeados a un decimal y por eso la suma no da exactamente 100 %.

El activo con más peso es BWX, con un 22 %. En 2012 la deuda pública de fuera de EE. UU., sobre todo la de Japón y la de Europa, valía más del doble que la del Tesoro estadounidense a más de un año. La renta variable suma un 40 % y la renta fija un 60 %, lejos de la cartera 60/40 clásica, que pone el 60 % en acciones. El mercado mundial tiene más bonos que acciones.

### Referencias

Doeswijk, R., Lam, T. y Swinkels, L. (2014). «The Global Multi-Asset Market Portfolio, 1959–2012». *Financial Analysts Journal*, 70(2), 26-41.

Doeswijk, R., Lam, T. y Swinkels, L. (2024). «Data update: The Global Multi-Asset Market Portfolio, 1959–2012». Erasmus University Rotterdam. Dataset. https://doi.org/10.25397/eur.9371741

iShares Trust (2013). Semi-Annual Report, iShares MSCI ACWI Index Fund, 31 de enero de 2013. Formulario N-CSRS presentado a la SEC. https://www.sec.gov/Archives/edgar/data/1100663/000119312513148033/d493068dncsrs.htm

U.S. Department of the Treasury (2012). Monthly Statement of the Public Debt of the United States, 31 de diciembre de 2012. https://fiscaldata.treasury.gov/datasets/monthly-statement-public-debt/

In [ ]:
# Pesos de mercado a cierre de 2012, calculados en el apartado 2
w_mkt = pd.Series({
    "SPY": 0.1826, "EFA": 0.1644, "EEM": 0.0511,
    "IEF": 0.0886, "TLT": 0.0150, "BWX": 0.2198,
    "LQD": 0.2027, "HYG": 0.0184, "TIP": 0.0249,
    "EMB": 0.0145, "EMLC": 0.0179,
})[TICKERS]
w_mkt = w_mkt / w_mkt.sum()   # el redondeo deja la suma en 0,9999

(w_mkt * 100).round(1).to_frame("w_mkt (%)")

## 3. Optimización inversa

Σ se estima con los retornos diarios de los tres últimos años, de octubre de 2023 a septiembre de 2026. Es la ventana del backtest, así que el análisis de hoy coincide con el último rebalanceo. Para δ usamos un Sharpe del mercado de 0,30, este sharpe de mercado lo hemos cogido del paper de Doeswijk, Lam y Swinkels (2020), el mismo estudio de donde hemos obtenido los pesos de la cartera de mercado. 

In [ ]:
from quantmgmt.optimizers.black_litterman import implied_returns, risk_aversion
from quantmgmt.optimizers.mean_variance import estimate_inputs

# Últimos tres años, la misma ventana que usará el último rebalanceo del backtest
ventana = rent.loc["2023-10-01":]
_, sigma = estimate_inputs(ventana, periods_per_year=252)

# Doeswijk, Lam y Swinkels (2020): 3,39 % de exceso con 11,2 % de volatilidad entre 1960 y 2017
SHARPE_MKT = 0.30
vol_mkt = float(np.sqrt(w_mkt @ sigma @ w_mkt))
delta = risk_aversion(w_mkt, sigma, SHARPE_MKT)
pi = implied_returns(w_mkt, sigma, delta)

print(f"Ventana: {ventana.index[0].date()} a {ventana.index[-1].date()} ({len(ventana)} días)")
print(f"Volatilidad del mercado: {vol_mkt:.2%}   δ = {delta:.2f}")
(pi * 100).round(2).to_frame("Π (%)")

In [ ]:
from quantmgmt.optimizers.mean_variance import max_sharpe

# Π con distintos Sharpe del mercado. El primero es el δ = 2,5 de He y Litterman (1999)
SHARPES = {
    f"{2.5 * vol_mkt:.2f}".replace(".", ",") + " (He y Litterman)": 2.5 * vol_mkt,
    "0,30 (Doeswijk)": 0.30,
    "0,40": 0.40,
    "0,50 (realizado 2010-2026)": 0.50,
}
sensibilidad = {}
for nombre, sharpe in SHARPES.items():
    d = risk_aversion(w_mkt, sigma, sharpe)
    pi_d = implied_returns(w_mkt, sigma, d)
    w_opt = max_sharpe(pi_d, sigma)
    print(f"Sharpe {nombre}: δ = {d:.2f}, diferencia máxima entre el máximo Sharpe y w_mkt = {(w_opt - w_mkt).abs().max():.4f}")
    sensibilidad[nombre] = pi_d * 100

pd.DataFrame(sensibilidad).round(2)

Para ver cuánto depende Π del Sharpe elegido, lo hemos calculado con cuatro valores. Van desde 0,21, que equivale al δ = 2,5 que usan He y Litterman, hasta 0,50, el Sharpe que ha tenido nuestra cartera de mercado entre 2010 y 2026. Al subir el Sharpe, δ sube en la misma proporción y todas las rentabilidades de la tabla suben a la vez. Con un Sharpe de 0,50, SPY exige un 5,9 % en lugar de un 3,6 %, y TIP un 1,3 % en lugar de un 0,8 %. Lo que no cambia es la relación entre activos. Cada uno sigue exigiendo lo mismo en proporción a los demás, y por eso la cartera de máximo Sharpe calculada con Π es en los cuatro casos la propia cartera de mercado.

Mientras no haya visiones, elegir un δ u otro no cambia la cartera. Cambiará cuando las añadamos en el apartado 4. Una visión se fija en puntos de rentabilidad, por ejemplo que SPY batirá a EFA en un 1 % anual. Ese 1 % pesa mucho si el equilibrio da rentabilidades del 2-3 % y bastante menos si da un 6-8 %. Con un δ más alto, las visiones mueven menos la cartera.

### Referencias

Doeswijk, R., Lam, T. y Swinkels, L. (2020). «Historical Returns of the Market Portfolio». *The Review of Asset Pricing Studies*, 10(3), 521-567.

He, G. y Litterman, R. (1999). «The Intuition Behind Black-Litterman Model Portfolios». Goldman Sachs Investment Management Research.

## 4. Visiones

Hemos formulado tres visiones y las tres son relativas. No tenemos una opinión sobre si el mercado en su conjunto subirá o bajará, solo sobre qué activos lo harán mejor que otros, así que preferimos no forzar ninguna visión absoluta.

| Visión | Estructura (P) | Q | Confianza | (1 − c) / c |
|---|---|---|---|---|
| 1. EE. UU. bate a los desarrollados | SPY − EFA | 4 % | 45 % | 1,22 |
| 2. Emergentes baten a EE. UU. | EEM − SPY | 2 % | 50 % | 1,00 |
| 3. Inflación por encima de lo esperado | TIP − IEF | 1 % | 70 % | 0,43 |

τ = 0,05 y Ω con el método de Idzorek, que explicamos más abajo.

La primera es que SPY batirá a EFA en un 4 % anual. Pensamos que EE. UU. es una economía más dinámica que la de los países desarrollados fuera de EE. UU. y que se va a beneficiar mucho más de la inteligencia artificial. Es la visión en la que menos confianza tenemos, un 45 %, porque el diferencial es grande y porque EFA no es solo Europa, también incluye Japón y Australia.

La segunda es que EEM batirá a SPY en un 2 % anual. La bolsa de EE. UU. cotiza con múltiplos por encima de su media histórica, mientras que China y buena parte de los emergentes siguen cotizando con múltiplos bajos. Esperamos una rotación del capital desde compañías caras hacia compañías más baratas pero con buenos fundamentales, algo parecido a un giro hacia el factor value. Le damos una confianza del 50 %.

Las dos primeras visiones no se contradicen. Creemos que EE. UU. crecerá más que el resto de desarrollados, pero ese crecimiento ya está en el precio, y los emergentes están baratos. Juntas implican que EEM batirá a EFA en torno a un 6 % anual, que es una apuesta grande, y por eso ninguna de las dos lleva una confianza alta.

La tercera es que TIP batirá a IEF en un 1 % anual, y es en la que más confianza tenemos, un 70 %. TIP e IEF tienen una duración parecida, así que la diferencia entre los dos es básicamente la inflación que se acabe dando menos la que descuenta hoy el mercado. Pensamos que la inflación se va a mantener por encima de lo esperado por las tensiones geopolíticas, el encarecimiento de las materias primas y, sobre todo, porque muchos gobiernos están muy endeudados y con déficit. Para ellos la inflación es una forma de reducir el peso real de su deuda, como ya pasó tras la pandemia. Estamos bastante seguros de la dirección, aunque no tanto de la magnitud.

En la tabla de debajo comparamos cada Q con lo que ya implica el equilibrio, P · Π. La visión se mide frente al equilibrio y no frente a cero. Si Q está por encima de lo que da Π, la visión es más optimista que el mercado y moverá la cartera hacia el activo que gana.

Para τ usamos 0,05, el valor de He y Litterman (1999). τ mide cuánto dudamos del equilibrio. Como nuestro Ω es proporcional a τ, los dos se escalan a la vez y τ casi no cambia la rentabilidad posterior, solo su incertidumbre.

Para Ω usamos el método de Idzorek (2005). Nos resulta más natural decir cuánta confianza tenemos en cada visión en porcentaje que dar directamente una varianza. La confianza c de cada visión se convierte en su varianza con ω = (1 − c) / c · P τΣ Pᵀ. Con un 50 % de confianza el factor vale uno y sale el Ω de He y Litterman. Con un 70 % el factor baja a 0,43 y la visión pesa más, y con un 45 % sube a 1,22 y pesa menos. Si la confianza fuera del 100 %, la visión se cumpliría exactamente, y si fuera casi nula, el modelo la ignoraría.

### Referencias

He, G. y Litterman, R. (1999). «The Intuition Behind Black-Litterman Model Portfolios». Goldman Sachs Investment Management Research.

Idzorek, T. (2005). «A step-by-step guide to the Black-Litterman model». Zephyr Associates.

In [ ]:
# Visiones: P (qué activos toca cada una), Q (cuánto, en decimal) y confianza (Idzorek)
P = pd.DataFrame(0.0, index=["V1", "V2", "V3"], columns=TICKERS)
P.loc["V1", "SPY"] = 1     # V1: SPY batirá a EFA
P.loc["V1", "EFA"] = -1
P.loc["V2", "EEM"] = 1     # V2: EEM batirá a SPY
P.loc["V2", "SPY"] = -1
P.loc["V3", "TIP"] = 1     # V3: TIP batirá a IEF
P.loc["V3", "IEF"] = -1

Q = pd.Series([0.04, 0.02, 0.01], index=P.index)
confianza = pd.Series([0.45, 0.50, 0.70], index=P.index)

# Lo que ya implica el equilibrio para cada visión: si Q es mayor, la visión es más optimista que el mercado
pd.DataFrame({"Q": Q, "Equilibrio (P·Π)": P @ pi, "Q - P·Π": Q - P @ pi}).round(4)

## 5. Posterior y pesos

# Parte B. HRP y contribución al riesgo

Implementación en `quantmgmt/optimizers/hrp.py`, desde cero: distancia sobre la matriz de correlaciones, clustering jerárquico (`scipy.cluster.hierarchy`), cuasi-diagonalización y bisección recursiva.

## 6. HRP

### Contribución al riesgo

# Parte C. Comparación

## 7. Backtest fuera de muestra

## Conclusión